# 1. Procesamiento ETL: RNPDNO

**Objetivo:** Descargar, limpiar, imputar CURPs y segmentar la base de datos del RNPDNO de manera automatizada.

In [ ]:
import sys
import os
import pandas as pd 
from datetime import datetime
from pathlib import Path
import importlib


# Configuración de rutas
# Agregamos el directorio raíz al sys.path para poder importar módulos de src/
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if project_root in sys.path:
    sys.path.remove(project_root)
sys.path.insert(0, project_root)

print(f"Raíz del proyecto configurada en: {project_root}")

import config

# Forzar modo local y recargar la configuración para usar el parquet verificado
os.environ["PIPELINE_MODO_INSUMOS"] = "local"
os.environ["PIPELINE_RNPDNO_MODO"] = "local"
os.environ["PIPELINE_RNPDNO_ARCHIVO"] = os.path.join(
    project_root,
    "input",
    "rnpdno",
    "reporte_personas_completo.parquet",
)

importlib.reload(config)
from src import cleaning
importlib.reload(cleaning)

### 1. El insumo del RNPDNO
De dónde sale el archivo lo decide **`config.MODO_INSUMOS`**, y los dos caminos son excluyentes: el que no se eligió no se intenta nunca, ni como respaldo del otro.

| Modo | Qué hace | Si falla |
|---|---|---|
| `"drive"` (por omisión) | Baja el Parquet del día de la carpeta de Drive y lo deja en `input/rnpdno/` con la fecha del corte en el nombre. Si ese corte ya está en disco con el mismo tamaño, no lo vuelve a bajar. | **Se detiene.** No corre con el Parquet que haya quedado en disco. |
| `"local"` | Lee el archivo exacto de `config.RNPDNO_LOCAL_FILE`, que puede ser el **Parquet** (`.parquet`) o el **Excel** de tres hojas (`.xlsx`). No toca Drive, no pide credenciales, no revisa la fecha del corte. | **Se detiene** diciendo qué ruta falta. |

Para correr sin descargar nada: cambia `MODO_INSUMOS = "local"` en `config.py` y apunta `RNPDNO_LOCAL_FILE` a tu archivo (o, sin editar nada, lanza Jupyter con `PIPELINE_MODO_INSUMOS=local`).

**El formato se reconoce por la extensión** (`config.formato_rnpdno`). Del Parquet, las tres hojas se reconstruyen con `estatus` + `conidentidad`; del Excel, se leen tal cual. A partir de la sección 2 el notebook ya no distingue de cuál vino: los dos dejan la misma tabla. Leer el Excel tarda alrededor de un minuto.

**En modo `"drive"`, el corte tiene que ser el de hoy.** Se verifica dos veces —con `modifiedTime` de Drive antes de bajar el archivo, y con `fecha_consulta` del propio Parquet después— y si no lo es, el proceso se detiene con `NO SE TIENE EL ARCHIVO DEL DÍA`. Es a propósito: evita que el reporte salga calculado sobre datos de otro día. En modo `"local"` esa revisión no aplica: la ruta se escribió a mano.

La descarga corre con tu cuenta de Google (OAuth). Las credenciales viven en `.secrets/`, fuera de git. Si es la primera vez, sigue `docs/9_credenciales_google_drive.md`: se abrirá el navegador una vez para que autorices el acceso de sólo lectura.

In [ ]:
# El insumo del RNPDNO: se baja de Drive o se lee de la ruta local, según
# .MODO_INSUMOS (ver src/drive.py). Los dos modos son excluyentes.
from src import drive
importlib.reload(drive)

print(f"⚙️  Modo de insumos: {config.RNPDNO_MODO_INSUMOS!r}\n")
drive.sincronizar_rnpdno()

# `config.RNPDNO_MANUAL_FILE_2` se resuelve al importar config, así que hay que
# recargarlo para que apunte al archivo recién bajado. A partir de aquí todo el
# notebook —y src/pipeline/exportacion_llaves.py— ve el mismo corte.
importlib.reload(config)

print(f"\n📄 Insumo del RNPDNO: {config.RNPDNO_MANUAL_FILE_2}")
# El formato se revisa aquí y no hasta leer el archivo: una extensión que no sea
# .parquet ni .xlsx se detiene antes de gastar tiempo.
formato_rnpdno = config.formato_rnpdno(config.RNPDNO_MANUAL_FILE_2)
print(f"   Formato: {formato_rnpdno}")
if formato_rnpdno == "parquet":
    print(f"   Corte declarado en el nombre: "
          f"{config.fecha_corte_parquet_rnpdno(config.RNPDNO_MANUAL_FILE_2)}")

In [ ]:
# El RNPDNO en el formato que tenga: Parquet (una tabla con las tres hojas apiladas)
# o Excel (las tres hojas). `leer_rnpdno` devuelve lo mismo en los dos casos —una
# sola tabla, con los nombres de columna del Parquet, todo como texto y los faltantes
# como nulo— y, aparte, la hoja de la que viene cada fila (ver src/lectura_rnpdno.py).
from src import lectura_rnpdno
importlib.reload(lectura_rnpdno)

rnpdno_crudo, hoja_origen = lectura_rnpdno.leer_rnpdno(config.RNPDNO_MANUAL_FILE_2)

print(f"Archivo: {config.RNPDNO_MANUAL_FILE_2.name} "
      f"({config.formato_rnpdno(config.RNPDNO_MANUAL_FILE_2)})")
print(f"Filas: {len(rnpdno_crudo):,} | Columnas: {rnpdno_crudo.shape[1]}")
print(pd.crosstab(rnpdno_crudo["estatus"], hoja_origen.rename("hoja")))
rnpdno_crudo.head(3)

In [ ]:
rnpdno_crudo.columns

### 2. Archivo de Entrada
Se separa la tabla de la sección anterior en las tres hojas del RNPDNO según `hoja_origen` (la hoja del Excel, o la que `estatus` + `conidentidad` reconstruyen en el Parquet) y se estandarizan sus columnas.

In [ ]:
# Construcción de `dfs`: un DataFrame por hoja, con las columnas objetivo
# estandarizadas y una copia intacta de cada original con sufijo _rnpdno. Sale igual
# del Parquet que del Excel: la celda anterior ya los dejó en la misma tabla.
from src.utils import standardize_columns # Importar nuestro validador estricto

# Usamos las columnas OPTIMIZADAS de config.py para filtrar la lectura
COLS_NECESARIAS_STRICT = [
    'fub', 'estatus', 'curp_vic', 'nombre_vic', 'primerapellido_vic',
    'idestado', 'fechahechos'
]
COLS_OPTIMIZADAS = set(config.RNPDNO_COLUMN_MAPPING.keys())

# --- Fecha de corte GLOBAL del archivo (independiente del estado y de la hoja) ---
# Se toma de `fecha_consulta`, que es la marca de tiempo con la que el RNPDNO generó
# la descarga: eso es el corte de la base. NO se usa `fechahechos`, que es la fecha
# del hecho más reciente capturado y puede ir semanas atrás del corte real
# (ACTUALIZACION_PENDIENTE, por ejemplo, arrastra hechos viejos).
max_fecha_archivo_global = pd.NaT

# Construir set de todos los nombres validos (estándar + variaciones)
valid_input_names = set()
for target in COLS_OPTIMIZADAS:
    valid_input_names.add(target)
    if target in config.RNPDNO_COLUMN_MAPPING:
        # Agregar variaciones normalizadas
        variations = [v.strip().lower().replace(' ', '_').replace('.', '') for v in config.RNPDNO_COLUMN_MAPPING[target]]
        valid_input_names.update(variations)

def filter_cols(col_name):
    """Equivalente al `usecols` que se aplicaba al leer el Excel."""
    if not isinstance(col_name, str): return False
    # Normalizar nombre de entrada para comparar
    norm = col_name.strip().lower().replace(' ', '_').replace('.', '')
    return norm in valid_input_names

print(f"DEBUG: Hojas requeridas: {list(config.SHEET_NAMES.values())}")

try:
    fuente = rnpdno_crudo
except NameError:
    raise NameError("Ejecuta primero la celda que carga `rnpdno_crudo`.")

dfs = {} # Inicializar el diccionario dfs
try:
    # 1. LAS HOJAS
    # `hoja_origen` viene de la celda anterior: la hoja del Excel, o la que
    # `estatus` + `conidentidad` reconstruyen en el Parquet.

    # Obtener ID del estado si hay filtro activo
    id_estado_filtro = None
    if hasattr(config, 'ESTADO') and config.ESTADO:
        id_estado_filtro = config.ESTADO
        nombre_estado = config.ENTIDADES.get(config.ESTADO, "Desconocido")
        print(f"⚙️ Filtro Estado Activo: {nombre_estado} -> ID: {id_estado_filtro}")

    for key, sheet_name in config.SHEET_NAMES.items():
        print(f"\nCargando hoja: {sheet_name} (key: {key})...")

        # --- Recorte de columnas, equivalente al 'usecols' del Excel ---
        print(f"  ⚡ Aplicando filtro de columnas...")
        df = fuente.loc[hoja_origen == key, [c for c in fuente.columns if filter_cols(c)]].copy()
        df = df.reset_index(drop=True)

        # Normalización inicial de nombres de columnas
        cols_limpias = [
            col.strip().lower().replace(' ', '_').replace('.', '')
            for col in df.columns
        ]
        df.columns = cols_limpias

        # --- REGLA B: Preservar columnas originales intactas con sufijo _rnpdno ---
        print(f"  📌 Preservando copias de columnas originales con sufijo _rnpdno...")
        for col in cols_limpias:
            if not col.endswith('_rnpdno'):
                df[f"{col}_rnpdno"] = df[col].copy()

        # Las claves llegan como texto: la copia `_rnpdno` las conserva así y la
        # columna de trabajo pasa a número, que es como la usan la deduplicación
        # y las derivadas.
        for _col in config.RNPDNO_PARQUET_COLS_NUMERICAS:
            if _col in df.columns:
                df[_col] = pd.to_numeric(df[_col], errors="coerce")

        # 2. ESTANDARIZACIÓN Y VALIDACIÓN DE COLUMNAS (Estricto)
        print(f"  🔍 Validando y estandarizando columnas...")
        try:
            df = standardize_columns(
                df,
                config.RNPDNO_COLUMN_MAPPING,
                strict=True,
                verbose=True
            )
            # Verificar manualmente las columnas NECESARIAS de config
            missing = [c for c in COLS_NECESARIAS_STRICT if c not in df.columns]
            if missing:
                raise ValueError(f"\n❌ ERROR CRÍTICO: La base de datos RNPDNO no contiene las siguientes columnas requeridas: {missing}.")

        except ValueError as e:
            print(f"  ❌ Falló la validación en hoja '{sheet_name}'.")
            raise e

        # --- RASTREAR FECHA DE CORTE GLOBAL ANTES DE FILTRAR POR ESTADO ---
        # Se compara contra las tres hojas para que el corte refleje TODA la base.
        if 'fecha_consulta' in df.columns:
            f_temp = pd.to_datetime(df['fecha_consulta'], errors='coerce')
            max_h = f_temp.max()
            if pd.notna(max_h):
                if pd.isna(max_fecha_archivo_global) or max_h > max_fecha_archivo_global:
                    max_fecha_archivo_global = max_h
                    print(f"  📅 Nueva fecha de consulta máxima detectada en hoja '{key}': {max_fecha_archivo_global}")
        else:
            print(f"  ⚠️ La hoja '{key}' no trae la columna 'fecha_consulta': no aporta al corte global.")

        # 3. VERIFICAR SI FUB EXISTE Y NO SE PERDIÓ
        if 'fub' not in df.columns:
             msg = f"❌ Error crítico: La columna 'fub' no se encontró tras el mapeo. Columnas presentes: {df.columns.tolist()}"
             print(msg)
             raise KeyError(msg)

        # --- APLICAR FILTRO POR ESTADO (Si existe en config) ---
        if id_estado_filtro is not None:
            col_entidad_id = 'idestado'

            if col_entidad_id in df.columns:
                print(f"  🔍 Filtrando por ID {id_estado_filtro} en columna '{col_entidad_id}'...")
                col_vals = pd.to_numeric(df[col_entidad_id], errors='coerce')
                mask = col_vals == id_estado_filtro

                registros_antes = len(df)
                df = df[mask].copy()
                registros_despues = len(df)
                print(f"    🔻 Registros: {registros_antes} -> {registros_despues} (Se eliminaron {registros_antes - registros_despues})")
            else:
                print(f"  ⚠️ Advertencia: No se encontró la columna '{col_entidad_id}' para filtrar por estado.")

        # Guardamos en diccionario
        dfs[key] = df
        print(f"  ✅ {len(df)} registros cargados y validados.")

except Exception as e:
     print(f"\n❌ ERROR FATAL DETENIENDO PROCESO: {e}")
     raise e
     

In [ ]:
# Vista previa de desaparecidas
if 'DESAPARECIDAS' in dfs:
    display(dfs['DESAPARECIDAS'].head(3))

In [ ]:
# Imprimir la fecha máxima en la columna 'fechahechos' de la hoja DESAPARECIDAS (si existe)
if 'DESAPARECIDAS' in dfs:
    if 'fechahechos' in dfs['DESAPARECIDAS'].columns:
        max_fecha = dfs['DESAPARECIDAS']['fechahechos'].dropna().max()
        print(f"📅 Fecha máxima en 'fechahechos' (DESAPARECIDAS): {max_fecha}")
    else:
        print("La columna 'fechahechos' no existe en la hoja DESAPARECIDAS.")
else:
    print("La hoja DESAPARECIDAS no está cargada en dfs.")

### 4. Verificación y Deduplicación
Verificamos que si existen FUBs duplicados, estos contengan información idéntica en los campos clave (Nombre, CURP, Sexo, etc.). Si es consistente, eliminamos duplicados. Si hay contradicciones, el proceso se detiene o alerta (lógica de `indicador_elementos_distintos`).

In [ ]:
# Definimos las columnas críticas que deben ser idénticas para una misma persona (FUB)
cols_identidad = [
    'curp_vic', 'nombre_vic', 'primerapellido_vic', 
    'segundoapellido_vic', 'sexo', 'idestadonacimiento_vic'
]

# Procesamos cada hoja cargada
dfs_limpios = {}

print("Iniciando verificación de duplicados...")

for key, df in dfs.items():
    print(f"\nProcesando hoja: {key}")
    
    # 1. Verificar consistencia
    try:
        # Aseguramos que las columnas existan, si no, usamos las que haya
        cols_presentes = [c for c in cols_identidad if c in df.columns]
        
        cleaning.verificar_consistencia_duplicados(df, key_col='fub', cols_to_check=cols_presentes)
        print("  ✅ Consistencia verificada.")
        
        # 2. Eliminar duplicados exactos de FUB
        df_limpio = cleaning.deduplicar_por_fub(df, key_col='fub')
        dfs_limpios[key] = df_limpio
        
    except ValueError as e:
        print(f"  ❌ Error crítico de validación: {e}")
        # En un script de producción, aquí podríamos detener todo con raise e

print("\nResumen:")
for key, df in dfs_limpios.items():
    print(f" - {key}: {len(df)} registros únicos.")

### 5. Cruce RENAPO e Imputación de CURPs
Buscamos recuperar CURPs faltantes o inválidas cruzando la información con bases auxiliares (RENAPO). Solo se reemplaza si la original es inválida y la nueva es válida. El resultado queda en `curp_vic_derivada_rnpdno`; la original se conserva intacta en `curp_vic_rnpdno`.

In [ ]:
# 1. Cargar y consolidar bases auxiliares de RENAPO
# La función busca los archivos configurados en config.RENAPO_FILES, los lee y
# deja una tabla con las columnas estándar: fub y curp_obtenida.
# Aunque aquí se muestra config.CRUCES_DIR, revisa cleaning.cargar_y_consolidar_curps:
# actualmente usa config.RENAPO_FILES para determinar los archivos que abre.
print(f"Buscando archivos RENAPO en: {config.CRUCES_DIR}")
curps_renapo = cleaning.cargar_y_consolidar_curps(config.CRUCES_DIR)

# Dentro de cargar_y_consolidar_curps, para cada archivo:
# - intenta encontrar las columnas cuyo nombre contenga 'fub' y 'curp';
# - valida la CURP con es_curp_valida_vectorizado y descarta las inválidas;
# - une los archivos y conserva una sola CURP por FUB (la primera encontrada).
# Si no encuentra archivos o CURPs válidas, devuelve una tabla vacía.

# 2. Aplicar imputación a todas las hojas cargadas
# dfs_limpios contiene los registros tras validar consistencia de FUB y deduplicar.
# Se crea un diccionario nuevo para almacenar el resultado por hoja sin reemplazar
# el diccionario dfs_limpios.
dfs_imputados = {}

print("\n--- Ejecutando imputación ---")
for key, df in dfs_limpios.items():
    # key es el nombre de la hoja, por ejemplo DESAPARECIDAS o LOCALIZADAS.
    print(f"\nProcesando hoja: {key}")
    
    # La imputación solo tiene sentido si la hoja tiene la columna original curp_vic.
    if 'curp_vic' in df.columns:
        # imputar_curps evalúa la CURP original; busca por FUB una CURP RENAPO
        # válida y la usa solo cuando la original no es válida.
        # No sobrescribe curp_vic: guarda el resultado en
        # curp_vic_derivada_rnpdno y calcula su bandera de validez en
        # curp_vic_valida_derivada_rnpdno. Si no hay coincidencia RENAPO,
        # la CURP derivada conserva el valor original.
        df_imp = cleaning.imputar_curps(df, curps_renapo)
        dfs_imputados[key] = df_imp
    else:
        # Sin curp_vic no se puede buscar ni imputar una CURP; se conserva el DataFrame.
        print("  ℹ️ No requiere imputación (no tiene campo curp_vic).")
        dfs_imputados[key] = df

print("\nProceso de imputación finalizado.")

### 5.5 Enriquecimiento de Variables
En este paso, directamente luego de la imputación, calculamos las columnas derivadas necesarias (como `nombre_completo_derivada_rnpdno`, `curp_vic_valida_derivada_rnpdno`, `fecha_hechos_modificada_derivada_rnpdno` y `estado_derivada_rnpdno`). Esto asegura que el archivo final mantenga todos los datos transformados para futuros análisis.

In [ ]:
from src.matching import normalizar_texto
from src.cleaning import es_curp_valida_vectorizado
import numpy as np

print("\n--- Enriquecimiento de variables (Variables Derivadas _derivada_rnpdno) ---")

# Enriquecemos tanto dfs_imputados como dfs_limpios
for dicc in [dfs_imputados, dfs_limpios]:
    for key, df in dicc.items():
        print(f"\nEnriqueciendo hoja: {key}")
        
        # 1. nombre_completo_derivada_rnpdno
        nombre = df.get('nombre_vic', df.get('nombre_vic_rnpdno', '')).fillna('').astype(str).replace('nan', '', regex=False)
        p_ap = df.get('primerapellido_vic', df.get('primerapellido_vic_rnpdno', '')).fillna('').astype(str).replace('nan', '', regex=False)
        s_ap = df.get('segundoapellido_vic', df.get('segundoapellido_vic_rnpdno', '')).fillna('').astype(str).replace('nan', '', regex=False)
        
        nom_raw = (nombre + " " + p_ap + " " + s_ap).str.replace(r'\s+', ' ', regex=True).str.strip()
        df['nombre_completo_derivada_rnpdno'] = normalizar_texto(nom_raw)
        
        # 2. CURP. La original queda intacta en `curp_vic_rnpdno` (REGLA B); la que decide
        # Desaparecidos/Pendientes es la imputada con RENAPO, que imputar_curps ya dejó
        # con su bandera en `dfs_imputados`. En `dfs_limpios` no se imputó nada: ahí la
        # imputada es la original y la bandera se calcula aquí, una sola vez.
        # No hay CURP "limpia" aparte: sería idéntica a la original, y el motor
        # estándar la limpia al comparar (src.cleaning.limpiar_curp).
        if 'curp_vic_derivada_rnpdno' not in df.columns:
            df['curp_vic_derivada_rnpdno'] = df['curp_vic'] if 'curp_vic' in df.columns else np.nan
            df['curp_vic_valida_derivada_rnpdno'] = es_curp_valida_vectorizado(
                df['curp_vic_derivada_rnpdno'].astype(str))
            
        # 3. Fechas. La copia `<columna>_rnpdno` conserva el texto tal como vino; la
        # convertida va a `<columna>_derivada_rnpdno`.
        for orig_c in ['fechahechos', 'fechanacimiento', 'fechapercato', 'fechacaptura', 'fechalocaliza']:
            df[f"{orig_c}_derivada_rnpdno"] = (pd.to_datetime(df[orig_c], errors='coerce')
                                              if orig_c in df.columns else pd.NaT)

        # fecha_hechos_modificada_derivada_rnpdno: hechos, o percato, o captura.
        df['fecha_hechos_modificada_derivada_rnpdno'] = (
            df['fechahechos_derivada_rnpdno']
            .fillna(df['fechapercato_derivada_rnpdno'])
            .fillna(df['fechacaptura_derivada_rnpdno'])
        )

        # 4. idestado_derivada_rnpdno, idmunicipio_derivada_rnpdno y estado_derivada_rnpdno
        src_idestado = 'idestado' if 'idestado' in df.columns else ('idestado_rnpdno' if 'idestado_rnpdno' in df.columns else None)
        if src_idestado:
            if hasattr(config, 'ENTIDADES'):
                ent_map = {k: v.upper() for k, v in config.ENTIDADES.items()}
            else:
                ent_map = {
                    1: 'AGUASCALIENTES', 2: 'BAJA CALIFORNIA', 3: 'BAJA CALIFORNIA SUR', 4: 'CAMPECHE',
                    5: 'COAHUILA DE ZARAGOZA', 6: 'COLIMA', 7: 'CHIAPAS', 8: 'CHIHUAHUA', 9: 'CIUDAD DE MEXICO',
                    10: 'DURANGO', 11: 'GUANAJUATO', 12: 'GUERRERO', 13: 'HIDALGO', 14: 'JALISCO',
                    15: 'ESTADO DE MEXICO', 16: 'MICHOACAN DE OCAMPO', 17: 'MORELOS', 18: 'NAYARIT', 19: 'NUEVO LEON',
                    20: 'OAXACA', 21: 'PUEBLA', 22: 'QUERETARO', 23: 'QUINTANA ROO', 24: 'SAN LUIS POTOSI',
                    25: 'SINALOA', 26: 'SONORA', 27: 'TABASCO', 28: 'TAMAULIPAS', 29: 'TLAXCALA',
                    30: 'VERACRUZ', 31: 'YUCATAN', 32: 'ZACATECAS', 33: 'NACIDO EN EL EXTRANJERO'
                }
            df['idestado_derivada_rnpdno'] = pd.to_numeric(df[src_idestado], errors='coerce')
            df['idmunicipio_derivada_rnpdno'] = pd.to_numeric(df['idmunicipio'], errors='coerce')
            df['estado_derivada_rnpdno'] = df['idestado_derivada_rnpdno'].map(ent_map)
            df['estado_derivada_rnpdno'] = normalizar_texto(df['estado_derivada_rnpdno'])
        else:
            df['idestado_derivada_rnpdno'] = np.nan
            df['idmunicipio_derivada_rnpdno'] = np.nan
            df['estado_derivada_rnpdno'] = None

        print(f"  ✅ Variables derivadas (_derivada_rnpdno) creadas exitosamente.")

### 6. Segmentación y Guardado
Finalmente, separamos la información en dos grupos:
1.  **Con CURP Válida**: Registros listos para análisis.
2.  **Actualización Pendiente**: Registros que requieren revisión manual o futura imputación.

Y generamos el Excel Final.

In [ ]:
# 1. Segmentación
print("Segmentando bases de datos...")
# Usamos dfs_imputados como base principal
# IMPORTANTE: R usa la lista ORIGINAL limpia ('comprimido_limpio') para la hoja LOCALIZADAS.
# En Python, 'dfs_limpios' equivale a 'comprimido_limpio'.
bases_segmentadas = cleaning.segmentar_bases(dfs_imputados, dfs_limpios)

# 2. Definición de ubicación de salida
# La fecha del reporte es el CORTE del RNPDNO: el máximo de `fecha_consulta` sobre las
# tres hojas, calculado en el paso de carga (`max_fecha_archivo_global`) antes del filtro
# por estado. Así refleja TODA la base y no una sola hoja ni una sola entidad.
# De aquí sale `fecha_consulta` en el maestro y, con ella, `fha_rn_*` en las PPT.
if pd.notna(max_fecha_archivo_global):
    fecha_reporte_str = max_fecha_archivo_global.strftime("%d%m%Y")
    print(f"🗓️ Corte del RNPDNO (máx. fecha_consulta de las 3 hojas): {fecha_reporte_str}")
else:
    # Fallback si ninguna hoja trajo `fecha_consulta` (no debería pasar).
    fecha_reporte_str = datetime.now().strftime("%d%m%Y")
    print(f"⚠️ Ninguna hoja trajo 'fecha_consulta'. Usando fecha de hoy: {fecha_reporte_str}")

# 3. Exportación (DESACTIVADA: Solo se generará el Parquet en el siguiente paso)
print("\n✅ ¡Segmentación completada exitosamente! (Saltando generación de Excel)")

### 4. Carga y Consolidación de Datos (Generación de Archivo Maestro)
Se unifican las tres clasificaciones (`DESAPARECIDAS`, `ACTUALIZACION_PENDIENTE`, `LOCALIZADAS`) en un solo DataFrame maestro de formato `.csv`, facilitando los calculos del paso 2 de Generación de Reportes.

In [ ]:
def consolidar_maestro_desde_memoria(bases_segmentadas, dfs_originales_limpios):
    """
    Unifica las tres clasificaciones (DESAPARECIDAS, ACTUALIZACION_PENDIENTE, LOCALIZADAS)
    directamente desde la memoria sin necesidad de leer un archivo Excel intermedio.
    Conserva columnas originales _rnpdno y columnas derivadas _derivada_rnpdno.
    """
    print("Iniciando consolidación de maestro desde memoria...")
    
    dfs = []

    # 1. DESAPARECIDAS (Registros con CURP válida)
    keys_des = [k for k in bases_segmentadas['con_curp'].keys() if k != 'LOCALIZADAS']
    lista_con_curp = [df for k, df in bases_segmentadas['con_curp'].items() if k in keys_des]
    if lista_con_curp:
        df_des = pd.concat(lista_con_curp, ignore_index=True)
        df_des['clasificacion_des_derivada_rnpdno'] = 'Desaparecidos'
        dfs.append(df_des)
        print(f"  ✅ DESAPARECIDAS: {len(df_des):,} registros unificados.")

    # 2. PENDIENTES (Registros sin CURP válida)
    lista_sin_curp = list(bases_segmentadas['sin_curp'].values())
    if lista_sin_curp:
        df_pend = pd.concat(lista_sin_curp, ignore_index=True)
        df_pend['clasificacion_des_derivada_rnpdno'] = 'Pendientes'
        dfs.append(df_pend)
        print(f"  ✅ PENDIENTES: {len(df_pend):,} registros unificados.")
        
    # 3. LOCALIZADAS (Universo de localizados)
    if 'LOCALIZADAS' in dfs_originales_limpios:
        df_loc = dfs_originales_limpios['LOCALIZADAS']
        df_loc['clasificacion_des_derivada_rnpdno'] = 'Localizados'
        dfs.append(df_loc)
        print(f"  ✅ LOCALIZADAS: {len(df_loc):,} registros unificados.")

    if not dfs:
        return pd.DataFrame()

    # Consolidar
    df_consolidado = pd.concat(dfs, ignore_index=True)

    # Corte de la base: una sola fecha para todo el archivo.
    fecha_dt_consulta = pd.to_datetime(fecha_reporte_str, format='%d%m%Y')
    df_consolidado['fecha_consulta_derivada_rnpdno'] = fecha_dt_consulta

    # Sólo sale el contrato de config.RNPDNO_OUTPUT_COLUMNS: las copias `_rnpdno`
    # tal como vinieron y las `_derivada_rnpdno`. Las columnas de trabajo sin sufijo
    # (`fub`, `curp_vic`, `fechahechos`…) se quedan en este notebook.
    df_consolidado = df_consolidado[config.RNPDNO_OUTPUT_COLUMNS]

    # --- HOMOGENEIZACIÓN DE TIPOS PARA PARQUET ---
    # Forzar homogeneización de tipos en todas las columnas object para PyArrow/Parquet
    print("🔨 Homogeneizando tipos en columnas object para PyArrow/Parquet...")
    for col_name in df_consolidado.columns:
        if df_consolidado[col_name].dtype == 'object':
            df_consolidado[col_name] = df_consolidado[col_name].astype(str).replace(r'^(nan|None|<NA>|NAN|NONE)$', np.nan, regex=True)
            df_consolidado[col_name] = df_consolidado[col_name].where(df_consolidado[col_name].notna() & (df_consolidado[col_name] != 'nan'), np.nan)

    return df_consolidado

try:
    df_maestro = consolidar_maestro_desde_memoria(bases_segmentadas, dfs_limpios)
    
    if df_maestro.empty:
        raise ValueError("❌ Error crítico: El DataFrame consolidado está vacío.")
        
    print(f"✅ Total de registros consolidados: {len(df_maestro):,}")
    print("Distribución por clasificación:")
    if 'clasificacion_des_derivada_rnpdno' in df_maestro.columns:
        print(df_maestro['clasificacion_des_derivada_rnpdno'].value_counts())
        
    # Exportar archivo Parquet
    # Nombre fijo declarado en config: es el contrato con los notebooks 3 y 4.
    maestro_path = config.ETL_RNPDNO_OUTPUT_FILE
    
    print(f"⌛ Exportando maestro consolidado a Parquet...")
    df_maestro.to_parquet(maestro_path, index=False, engine='pyarrow')
    print(f"🎉 Maestro consolidado exportado correctamente en: {maestro_path}")
    
except Exception as e:
    print(f"\n⛔ Ocurrió un error en la consolidación del maestro: {e}")
    raise e